<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="http://mng.bz/orYv">《从零构建大语言模型》（Build a Large Language Model From Scratch）</a> 一书的配套代码，作者 <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库：<a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 从零实现字节对编码（BPE）分词器

- 本 notebook 从零实现流行的字节对编码（BPE）分词算法，用于教学；GPT-2 至 GPT-4、Llama 3 等模型均使用 BPE
- 分词目的详见 [第 2 章](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/01_main-chapter-code/ch02.ipynb)；此处为解释 BPE 算法的 bonus 材料
- OpenAI 为原始 GPT 模型实现的 BPE 分词器见 [此处](https://github.com/openai/gpt-2/blob/master/src/encoder.py)
- BPE 算法最初由 Philip Gage 于 1994 年提出："[A New Algorithm for Data Compression](https://github.com/tpn/pdfs/blob/master/A%20New%20Algorithm%20for%20Data%20Compression%20(1994).pdf)"
- 如今包括 Llama 3 在内的大多数项目使用 OpenAI 开源 [tiktoken](https://github.com/openai/tiktoken)，因其计算性能高；可加载预训练 GPT-2/GPT-4 分词器（Llama 3 也使用 GPT-4 分词器训练）
- 与上述实现不同，本 notebook 的实现还包含训练分词器的函数（教学用途）
- 另有 [minBPE](https://github.com/karpathy/minbpe) 支持训练，可能更高效（本实现侧重教学）；相比 `minbpe`，本实现还可加载原始 OpenAI 词表与 BPE "merges"（此外 Hugging Face 分词器也支持训练与加载；读者在 [GitHub 讨论](https://github.com/rasbt/LLMs-from-scratch/discussions/485) 中分享了尼泊尔语 BPE 训练经验）

&nbsp;
# 1. 字节对编码（BPE）的核心思想

- BPE 的核心思想是将文本转为整数表示（token ID）供 LLM 训练（见 [第 2 章](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/01_main-chapter-code/ch02.ipynb))

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/bpe-from-scratch/bpe-overview.webp" width="600px">

&nbsp;
## 1.1 位与字节

- 在介绍 BPE 算法前，先说明字节（byte）的概念
- 考虑将文本转为字节数组（BPE 即 "byte" pair encoding）：

In [1]:
text = "This is some text"
byte_ary = bytearray(text, "utf-8")
print(byte_ary)

bytearray(b'This is some text')


- 对 `bytearray` 调用 `list()` 时，每个字节作为独立元素，结果为对应字节值的整数列表：

In [2]:
ids = list(byte_ary)
print(ids)

[84, 104, 105, 115, 32, 105, 115, 32, 115, 111, 109, 101, 32, 116, 101, 120, 116]


- 这可将文本转为 LLM 嵌入层所需的 token ID 表示
- 但缺点是每个字符一个 ID（短文本也会产生很多 ID！）
- 即 17 字符输入需 17 个 token ID 作为 LLM 输入：

In [3]:
print("Number of characters:", len(text))
print("Number of token IDs:", len(ids))

Number of characters: 17
Number of token IDs: 17


- 若接触过 LLM，可知 BPE 分词器词表包含整词或子词的 token ID，而非每字符一个
- 例如 GPT-2 将 "This is some text" 分词为 4 个而非 17 个 token：`1212, 318, 617, 2420`
- 可用交互式 [tiktoken app](https://tiktokenizer.vercel.app/?model=gpt2) 或 [tiktoken 库](https://github.com/openai/tiktoken) 验证：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/bpe-from-scratch/tiktokenizer.webp" width="600px">

```python
import tiktoken

gpt2_tokenizer = tiktoken.get_encoding("gpt2")
gpt2_tokenizer.encode("This is some text")
# prints [1212, 318, 617, 2420]
```

- 一个字节 8 位，共 2<sup>8</sup> = 256 种取值（0–255）
- 执行 `bytearray(range(0, 257))` 会警告 `ValueError: byte must be in range(0, 256)`
- BPE 分词器通常将这 256 值作为前 256 个单字符 token；可用以下代码查看：

```python
import tiktoken
gpt2_tokenizer = tiktoken.get_encoding("gpt2")

for i in range(300):
    decoded = gpt2_tokenizer.decode([i])
    print(f"{i}: {decoded}")
"""
prints:
0: !
1: "
2: #
...
255: �  # <---- single character tokens up to here
256:  t
257:  a
...
298: ent
299:  n
"""
```

- 注意 256、257 项不是单字符而是双字符（空格+字母），这是原始 GPT-2 BPE 的小缺陷（GPT-4 分词器已改进）

&nbsp;
## 1.2 构建词表

- BPE 分词算法目标是构建常见子词词表，如 `298: ent`（出现在 *entangle, entertain, enter, entrance, entity, ...* 等词中），甚至完整词如 

```
318: is
617: some
1212: This
2420: text
```

- BPE 算法最初由 Philip Gage 于 1994 年提出："[A New Algorithm for Data Compression](https://github.com/tpn/pdfs/blob/master/A%20New%20Algorithm%20for%20Data%20Compression%20(1994).pdf)"
- 在进入代码实现前，先总结当今 LLM 分词器使用的 BPE 形式（见以下各节）：

&nbsp;
## 1.3 BPE 算法概要

**1. 识别高频对**
- 每轮扫描文本，找出最常见的字节（或字符）对

**2. 替换并记录**

- 将该对替换为新占位 ID（尚未使用的 ID，如从 0…255 开始则首个为 256）
- 在查找表中记录映射
- 查找表大小为超参数，即「词表大小」（GPT-2 为 50,257）

**3. 重复直至无收益**

- 重复步骤 1、2，持续合并最高频对
- 无法进一步压缩时停止（如每对最多出现一次）

**解压缩（解码）**

- 用查找表将每个 ID 替换回对应字符对，逆序还原原文



&nbsp;
## 1.4 BPE 算法示例

### 1.4.1 编码部分的具体示例（1.3 节步骤 1 和 2）

- 假设训练文本为 `the cat in the hat`，要为其构建 BPE 词表

**第 1 轮迭代**

1. 识别高频对
  - 文本中 "th" 出现两次（开头及第二个 "e" 前）

2. 替换并记录
  - 将 "th" 替换为未使用的 token ID，如 256
  - 新文本：`<256>e cat in <256>e hat`
  - 新词表：

```
  0: ...
  ...
  256: "th"
```

**第 2 轮迭代**

1. **识别高频对**  
   - 在 `<256>e cat in <256>e hat` 中，`<256>e` 出现两次

2. **替换并记录**  
   - 将 `<256>e` 替换为新 ID，如 `257`  
   - 新文本：
     ```
     <257> cat in <257> hat
     ```
   - 更新词表：
     ```
     0: ...
     ...
     256: "th"
     257: "<256>e"
     ```

**第 3 轮迭代**

1. **识别高频对**  
   - 在 `<257> cat in <257> hat` 中，`<257> ` 出现两次（开头及 “hat” 前）

2. **替换并记录**  
   - 将 `<257> ` 替换为新 ID，如 `258`  
   - 新文本：
     ```
     <258>cat in <258>hat
     ```
   - 更新词表：
     ```
     0: ...
     ...
     256: "th"
     257: "<256>e"
     258: "<257> "
     ```
     
- 依此类推

&nbsp;
### 1.4.2 解码部分的具体示例（1.3 节步骤 3）

- 还原原文：按引入顺序的逆序，将每个 token ID 替换回对应字符对
- 从最终压缩文本开始：`<258>cat in <258>hat`
- 替换 `<258>` → `<257> `：`<257> cat in <257> hat`  
- 替换 `<257>` → `<256>e`：`<256>e cat in <256>e hat`
- 替换 `<256>` → "th"：`the cat in the hat`

&nbsp;
## 2. 简易 BPE 实现

- 下面将实现上述算法的 Python 类，接口类似 `tiktoken`
- 上文编码部分描述 `train()` 的训练步骤；`encode()` 类似（因特殊 token 处理而更复杂）：

1. 将输入文本拆分为单个字节
2. 反复查找并合并相邻 token 对（与已学 BPE merges 匹配，按 rank 从高到低，即学习顺序）
3. 直至无法继续合并
4. 最终 token ID 列表即为编码结果

In [4]:
from collections import Counter, deque
from functools import lru_cache
import re
import json


class BPETokenizerSimple:
    def __init__(self):
        # Maps token_id to token_str (e.g., {11246: "some"})
        self.vocab = {}
        # Maps token_str to token_id (e.g., {"some": 11246})
        self.inverse_vocab = {}
        # Dictionary of BPE merges: {(token_id1, token_id2): merged_token_id}
        self.bpe_merges = {}

        # For the official OpenAI GPT-2 merges, use a rank dict:
        #  of form {(string_A, string_B): rank}, where lower rank = higher priority
        self.bpe_ranks = {}

    def train(self, text, vocab_size, allowed_special={"<|endoftext|>"}):
        """
        Train the BPE tokenizer from scratch.

        Args:
            text (str): The training text.
            vocab_size (int): The desired vocabulary size.
            allowed_special (set): A set of special tokens to include.
        """

        # Pre-tokenize training text using the same boundary rules as encode()
        tokens = self.pretokenize_text(text)

        # Initialize vocab with unique characters, including "Ġ" if present
        # Start with the first 256 ASCII characters
        unique_chars = [chr(i) for i in range(256)]
        unique_chars.extend(
            char for char in sorted({char for token in tokens for char in token})
            if char not in unique_chars
        )
        if "Ġ" not in unique_chars:
            unique_chars.append("Ġ")

        self.vocab = {i: char for i, char in enumerate(unique_chars)}
        self.inverse_vocab = {char: i for i, char in self.vocab.items()}

        # Add allowed special tokens
        if allowed_special:
            for token in allowed_special:
                if token not in self.inverse_vocab:
                    new_id = len(self.vocab)
                    self.vocab[new_id] = token
                    self.inverse_vocab[token] = new_id

        # Tokenize each pre-token into character IDs
        token_id_sequences = [
            [self.inverse_vocab[char] for char in token]
            for token in tokens
        ]

        # BPE steps 1-3: Repeatedly find and replace frequent pairs
        for new_id in range(len(self.vocab), vocab_size):
            pair_id = self.find_freq_pair(token_id_sequences, mode="most")
            if pair_id is None:
                break
            token_id_sequences = self.replace_pair(token_id_sequences, pair_id, new_id)
            self.bpe_merges[pair_id] = new_id

        # Build the vocabulary with merged tokens
        for (p0, p1), new_id in self.bpe_merges.items():
            merged_token = self.vocab[p0] + self.vocab[p1]
            self.vocab[new_id] = merged_token
            self.inverse_vocab[merged_token] = new_id

    def load_vocab_and_merges_from_openai(self, vocab_path, bpe_merges_path):
        """
        Load pre-trained vocabulary and BPE merges from OpenAI's GPT-2 files.

        Args:
            vocab_path (str): Path to the vocab file (GPT-2 calls it 'encoder.json').
            bpe_merges_path (str): Path to the bpe_merges file  (GPT-2 calls it 'vocab.bpe').
        """
        # Load vocabulary
        with open(vocab_path, "r", encoding="utf-8") as file:
            loaded_vocab = json.load(file)
            # encoder.json is {token_str: id}; we want id->str and str->id
            self.vocab = {int(v): k for k, v in loaded_vocab.items()}
            self.inverse_vocab = {k: int(v) for k, v in loaded_vocab.items()}
    
        # Must have GPT-2's printable newline character 'Ċ' (U+010A) at id 198
        if "Ċ" not in self.inverse_vocab or self.inverse_vocab["Ċ"] != 198:
            raise KeyError("Vocabulary missing GPT-2 newline glyph 'Ċ' at id 198.")
    
        # Must have <|endoftext|> at 50256
        if "<|endoftext|>" not in self.inverse_vocab or self.inverse_vocab["<|endoftext|>"] != 50256:
            raise KeyError("Vocabulary missing <|endoftext|> at id 50256.")
    
        # Provide a convenience alias for '\n' -> 198
        # Keep printable character 'Ċ' in vocab so BPE merges keep working
        if "\n" not in self.inverse_vocab:
            self.inverse_vocab["\n"] = self.inverse_vocab["Ċ"]

        if "\r" not in self.inverse_vocab:
            if 201 in self.vocab:
                self.inverse_vocab["\r"] = 201
            else:
                raise KeyError("Vocabulary missing carriage return token at id 201.")

        # Load GPT-2 merges and store ranks
        self.bpe_ranks = {}
        with open(bpe_merges_path, "r", encoding="utf-8") as file:
            lines = file.readlines()
            if lines and lines[0].startswith("#"):
                lines = lines[1:]
    
            rank = 0
            for line in lines:
                token1, *rest = line.strip().split()
                if len(rest) != 1:
                    continue
                token2 = rest[0]
                if token1 in self.inverse_vocab and token2 in self.inverse_vocab:
                    self.bpe_ranks[(token1, token2)] = rank
                    rank += 1
                else:
                    # Safe to skip pairs whose symbols are not in vocab
                    pass


    def encode(self, text, allowed_special=None):
        """
        Encode the input text into a list of token IDs, with tiktoken-style handling of special tokens.
    
        Args:
            text (str): The input text to encode.
            allowed_special (set or None): Special tokens to allow passthrough. If None, special handling is disabled.
    
        Returns:
            List of token IDs.
        """
    
        # ---- This section is to mimic tiktoken in terms of allowed special tokens ----
        specials_in_vocab = [
            tok for tok in self.inverse_vocab
            if tok.startswith("<|") and tok.endswith("|>")
        ]
        if allowed_special is None:
            # Nothing is allowed
            disallowed = [tok for tok in specials_in_vocab if tok in text]
            if disallowed:
                raise ValueError(f"Disallowed special tokens encountered in text: {disallowed}")
        else:
            # Some spefic tokens are allowed (e.g., we use this for <|endoftext|>)
            disallowed = [tok for tok in specials_in_vocab if tok in text and tok not in allowed_special]
            if disallowed:
                raise ValueError(f"Disallowed special tokens encountered in text: {disallowed}")
        # -----------------------------------------------------------------------------

        token_ids = []
        # If some specials are allowed, split around them and passthrough those ids
        if allowed_special is not None and len(allowed_special) > 0:
            special_pattern = "(" + "|".join(
                re.escape(tok) for tok in sorted(allowed_special, key=len, reverse=True)
            ) + ")"
    
            last_index = 0
            for match in re.finditer(special_pattern, text):
                prefix = text[last_index:match.start()]
                token_ids.extend(self.encode(prefix, allowed_special=None))  # encode prefix normally
    
                special_token = match.group(0)
                if special_token in self.inverse_vocab:
                    token_ids.append(self.inverse_vocab[special_token])
                else:
                    raise ValueError(f"Special token {special_token} not found in vocabulary.")
                last_index = match.end()
    
            text = text[last_index:]  # remainder to process normally
    
            # Extra guard for any other special literals left over
            disallowed = [
                tok for tok in self.inverse_vocab
                if tok.startswith("<|") and tok.endswith("|>") and tok in text and tok not in allowed_special
            ]
            if disallowed:
                raise ValueError(f"Disallowed special tokens encountered in text: {disallowed}")

    
        # ---- Newline and carriage return handling ----
        tokens = self.pretokenize_text(text)
        # ---------------------------------------------------------------
    
        # Map tokens -> ids (BPE if needed)
        for tok in tokens:
            if tok in self.inverse_vocab:
                token_ids.append(self.inverse_vocab[tok])
            else:
                token_ids.extend(self.tokenize_with_bpe(tok))
    
        return token_ids

    def tokenize_with_bpe(self, token):
        """
        Tokenize a single token using BPE merges.

        Args:
            token (str): The token to tokenize.

        Returns:
            List[int]: The list of token IDs after applying BPE.
        """
        # Tokenize the token into individual characters (as initial token IDs)
        token_ids = [self.inverse_vocab.get(char, None) for char in token]
        if None in token_ids:
            missing_chars = [char for char, tid in zip(token, token_ids) if tid is None]
            raise ValueError(f"Characters not found in vocab: {missing_chars}")

        # If we haven't loaded OpenAI's GPT-2 merges, use my approach
        if not self.bpe_ranks:
            can_merge = True
            while can_merge and len(token_ids) > 1:
                can_merge = False
                new_tokens = []
                i = 0
                while i < len(token_ids) - 1:
                    pair = (token_ids[i], token_ids[i + 1])
                    if pair in self.bpe_merges:
                        merged_token_id = self.bpe_merges[pair]
                        new_tokens.append(merged_token_id)
                        # Uncomment for educational purposes:
                        # print(f"Merged pair {pair} -> {merged_token_id} ('{self.vocab[merged_token_id]}')")
                        i += 2  # Skip the next token as it's merged
                        can_merge = True
                    else:
                        new_tokens.append(token_ids[i])
                        i += 1
                if i < len(token_ids):
                    new_tokens.append(token_ids[i])
                token_ids = new_tokens
            return token_ids

        # Otherwise, do GPT-2-style merging with the ranks:
        # 1) Convert token_ids back to string "symbols" for each ID
        symbols = [self.vocab[id_num] for id_num in token_ids]

        # Repeatedly merge all occurrences of the lowest-rank pair
        while True:
            # Collect all adjacent pairs
            pairs = set(zip(symbols, symbols[1:]))
            if not pairs:
                break

            # Find the pair with the best (lowest) rank
            min_rank = float("inf")
            bigram = None
            for p in pairs:
                r = self.bpe_ranks.get(p, float("inf"))
                if r < min_rank:
                    min_rank = r
                    bigram = p

            # If no valid ranked pair is present, we're done
            if bigram is None or bigram not in self.bpe_ranks:
                break

            # Merge all occurrences of that pair
            first, second = bigram
            new_symbols = []
            i = 0
            while i < len(symbols):
                # If we see (first, second) at position i, merge them
                if i < len(symbols) - 1 and symbols[i] == first and symbols[i+1] == second:
                    new_symbols.append(first + second)  # merged symbol
                    i += 2
                else:
                    new_symbols.append(symbols[i])
                    i += 1
            symbols = new_symbols

            if len(symbols) == 1:
                break

        # Finally, convert merged symbols back to IDs
        merged_ids = [self.inverse_vocab[sym] for sym in symbols]
        return merged_ids

    def decode(self, token_ids):
        """
        Decode a list of token IDs back into a string.

        Args:
            token_ids (List[int]): The list of token IDs to decode.

        Returns:
            str: The decoded string.
        """
        out = []
        for tid in token_ids:
            if tid not in self.vocab:
                raise ValueError(f"Token ID {tid} not found in vocab.")
            tok = self.vocab[tid]

            # Map GPT-2 special chars back to real chars
            if tid == 198 or tok == "\n":
                out.append("\n")
            elif tid == 201 or tok == "\r":
                out.append("\r")
            elif tok.startswith("Ġ"):
                out.append(" " + tok[1:])
            else:
                out.append(tok)
        return "".join(out)

    def save_vocab_and_merges(self, vocab_path, bpe_merges_path):
        """
        Save the vocabulary and BPE merges to JSON files.

        Args:
            vocab_path (str): Path to save the vocabulary.
            bpe_merges_path (str): Path to save the BPE merges.
        """
        # Save vocabulary
        with open(vocab_path, "w", encoding="utf-8") as file:
            json.dump(self.vocab, file, ensure_ascii=False, indent=2)

        # Save BPE merges as a list of dictionaries
        with open(bpe_merges_path, "w", encoding="utf-8") as file:
            merges_list = [{"pair": list(pair), "new_id": new_id}
                           for pair, new_id in self.bpe_merges.items()]
            json.dump(merges_list, file, ensure_ascii=False, indent=2)

    def load_vocab_and_merges(self, vocab_path, bpe_merges_path):
        """
        Load the vocabulary and BPE merges from JSON files.

        Args:
            vocab_path (str): Path to the vocabulary file.
            bpe_merges_path (str): Path to the BPE merges file.
        """
        # Load vocabulary
        with open(vocab_path, "r", encoding="utf-8") as file:
            loaded_vocab = json.load(file)
            self.vocab = {int(k): v for k, v in loaded_vocab.items()}
            self.inverse_vocab = {v: int(k) for k, v in loaded_vocab.items()}

        # Load BPE merges
        with open(bpe_merges_path, "r", encoding="utf-8") as file:
            merges_list = json.load(file)
            for merge in merges_list:
                pair = tuple(merge["pair"])
                new_id = merge["new_id"]
                self.bpe_merges[pair] = new_id

    @lru_cache(maxsize=None)
    def get_special_token_id(self, token):
        return self.inverse_vocab.get(token, None)

    @staticmethod
    def pretokenize_text(text):
        tokens = []
        parts = re.split(r'(\r\n|\r|\n)', text)
        for part in parts:
            if part == "":
                continue
            if part == "\r\n":
                tokens.append("\r")
                tokens.append("\n")
                continue
            if part == "\r":
                tokens.append("\r")
                continue
            if part == "\n":
                tokens.append("\n")
                continue

            # Normal chunk without line breaks:
            # - If spaces precede a word, prefix the first word with 'Ġ' and
            #   add standalone 'Ġ' for additional spaces
            # - If spaces trail the chunk (e.g., before a newline) add
            #   standalone 'Ġ' tokens (tiktoken produces id 220 for 'Ġ')
            pending_spaces = 0
            for m in re.finditer(r'( +)|(\S+)', part):
                if m.group(1) is not None:
                    pending_spaces += len(m.group(1))
                else:
                    word = m.group(2)
                    if pending_spaces > 0:
                        for _ in range(pending_spaces - 1):
                            tokens.append("Ġ")  # remaining spaces as standalone
                        tokens.append("Ġ" + word) # one leading space
                        pending_spaces = 0
                    else:
                        tokens.append(word)
            # Trailing spaces (no following word): add standalone 'Ġ' tokens
            for _ in range(pending_spaces):
                tokens.append("Ġ")
        return tokens

    @staticmethod
    def find_freq_pair(token_id_sequences, mode="most"):
        pairs = Counter(
            pair
            for token_ids in token_id_sequences
            for pair in zip(token_ids, token_ids[1:])
        )

        if not pairs:
            return None

        if mode == "most":
            return max(pairs.items(), key=lambda x: x[1])[0]
        elif mode == "least":
            return min(pairs.items(), key=lambda x: x[1])[0]
        else:
            raise ValueError("Invalid mode. Choose 'most' or 'least'.")

    @staticmethod
    def replace_pair(token_id_sequences, pair_id, new_id):
        replaced_sequences = []

        for token_ids in token_id_sequences:
            dq = deque(token_ids)
            replaced = []

            while dq:
                current = dq.popleft()
                if dq and (current, dq[0]) == pair_id:
                    replaced.append(new_id)
                    # Remove the 2nd token of the pair, 1st was already removed
                    dq.popleft()
                else:
                    replaced.append(current)

            replaced_sequences.append(replaced)

        return replaced_sequences

- 上述 `BPETokenizerSimple` 类代码较多，本 notebook 不逐行讨论，下一节简要介绍用法以理解类方法

## 3. BPE 实现走读

- 实践中强烈建议使用 [tiktoken](https://github.com/openai/tiktoken)；本实现侧重可读性与教学，非性能
- 用法与 tiktoken 大致相似，但 tiktoken 无训练方法
- 下面通过示例了解 `BPETokenizerSimple` 如何工作（不展开代码细节）

### 3.1 训练、编码与解码

- 首先用一些样本文本作为训练数据集：

In [5]:
import os
import requests

def download_file_if_absent(url, filename, search_dirs):
    for directory in search_dirs:
        file_path = os.path.join(directory, filename)
        if os.path.exists(file_path):
            print(f"{filename} already exists in {file_path}")
            return file_path

    target_path = os.path.join(search_dirs[0], filename)
    try:
        response = requests.get(url, stream=True, timeout=60)
        response.raise_for_status()
        with open(target_path, "wb") as out_file:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    out_file.write(chunk)
        print(f"Downloaded {filename} to {target_path}")
    except Exception as e:
        print(f"Failed to download {filename}. Error: {e}")

    return target_path


verdict_path = download_file_if_absent(
    url=(
         "https://raw.githubusercontent.com/rasbt/"
         "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
         "the-verdict.txt"
    ),
    filename="the-verdict.txt",
    search_dirs=["ch02/01_main-chapter-code/", "../01_main-chapter-code/", "."]
)

with open(verdict_path, "r", encoding="utf-8") as f: # added ../01_main-chapter-code/
    text = f.read()

the-verdict.txt already exists in ../01_main-chapter-code/the-verdict.txt


- 接下来初始化并训练词表大小为 1,000 的 BPE 分词器
- 词表默认已有 256 项（字节值），另计 `<|endoftext|>` 与 `Ġ` 空白 token，实际学习 742 项
- 对比：GPT-2 词表 50,257；GPT-4 为 100,256（tiktoken 的 `cl100k_base`）；GPT-4o 为 199,997（`o200k_base`）；它们训练集远大于我们的示例文本

In [6]:
tokenizer = BPETokenizerSimple()
tokenizer.train(text, vocab_size=1000, allowed_special={"<|endoftext|>"})

- 可查看词表内容（注意会输出很长列表）

In [7]:
# print(tokenizer.vocab)
print(len(tokenizer.vocab))

1000


- 该词表通过 742 次合并创建（`= 1000 - len(range(0, 256)) - len(special_tokens) - "Ġ" = 1000 - 256 - 1 - 1 = 742`）

In [8]:
print(len(tokenizer.bpe_merges))

742


- 即前 256 项为单字符 token

- 接下来用 `encode` 方法及已学习的 merges 编码文本：

In [9]:
input_text = "Jack embraced beauty through art and life."
token_ids = tokenizer.encode(input_text)
print(token_ids)

[424, 256, 654, 531, 302, 311, 256, 296, 97, 465, 121, 595, 841, 116, 287, 466, 256, 326, 972, 46]


In [10]:
input_text = "Jack embraced beauty through art and life.<|endoftext|> "
token_ids = tokenizer.encode(input_text, allowed_special={"<|endoftext|>"})
print(token_ids)

[424, 256, 654, 531, 302, 311, 256, 296, 97, 465, 121, 595, 841, 116, 287, 466, 256, 326, 972, 46, 257, 256]


In [11]:
print("Number of characters:", len(input_text))
print("Number of token IDs:", len(token_ids))

Number of characters: 56
Number of token IDs: 22


- 从长度可见，42 字符句子编码为 20 个 token ID，相比逐字符编码约减半

- `decode()` 使用词表将 token ID 映射回文本：

In [12]:
print(token_ids)

[424, 256, 654, 531, 302, 311, 256, 296, 97, 465, 121, 595, 841, 116, 287, 466, 256, 326, 972, 46, 257, 256]


In [13]:
print(tokenizer.decode(token_ids))

Jack embraced beauty through art and life.<|endoftext|> 


- 逐 token ID 迭代可更好理解解码过程：

In [14]:
for token_id in token_ids:
    print(f"{token_id} -> {tokenizer.decode([token_id])}")

424 -> Jack
256 ->  
654 -> em
531 -> br
302 -> ac
311 -> ed
256 ->  
296 -> be
97 -> a
465 -> ut
121 -> y
595 ->  through
841 ->  ar
116 -> t
287 ->  a
466 -> nd
256 ->  
326 -> li
972 -> fe
46 -> .
257 -> <|endoftext|>
256 ->  


- 可见多数 token ID 表示 2 字符子词；因训练文本短、重复词少，且词表较小

- 总结：`decode(encode())` 应能还原任意输入文本：

In [15]:
tokenizer.decode(
    tokenizer.encode("This is some text.")
)

'This is some text.'

In [16]:
tokenizer.decode(
    tokenizer.encode("This is some text with \n newline characters.")
)

'This is some text with \n newline characters.'

### 3.2 保存与加载分词器

- 接下来看如何保存训练好的分词器以便复用：

In [17]:
# Save trained tokenizer
tokenizer.save_vocab_and_merges(vocab_path="vocab.json", bpe_merges_path="bpe_merges.txt")

In [18]:
# Load tokenizer
tokenizer2 = BPETokenizerSimple()
tokenizer2.load_vocab_and_merges(vocab_path="vocab.json", bpe_merges_path="bpe_merges.txt")

- 加载后的分词器应产生与之前相同的结果：

In [19]:
print(tokenizer2.decode(token_ids))

Jack embraced beauty through art and life.<|endoftext|> 


In [20]:
tokenizer2.decode(
    tokenizer2.encode("This is some text with \n newline characters.")
)

'This is some text with \n newline characters.'

&nbsp;
### 3.3 加载 OpenAI 原始 GPT-2 BPE 分词器

- 最后加载 OpenAI 的 GPT-2 分词器文件

In [21]:
# Download files if not already present in this directory

# Define the directories to search and the files to download
search_directories = ["ch02/02_bonus_bytepair-encoder/gpt2_model/", "../02_bonus_bytepair-encoder/gpt2_model/", "."]

files_to_download = {
    "https://openaipublic.blob.core.windows.net/gpt-2/models/124M/vocab.bpe": "vocab.bpe",
    "https://openaipublic.blob.core.windows.net/gpt-2/models/124M/encoder.json": "encoder.json"
}

# Ensure directories exist and download files if needed
paths = {}
for url, filename in files_to_download.items():
    paths[filename] = download_file_if_absent(url, filename, search_directories)

vocab.bpe already exists in ../02_bonus_bytepair-encoder/gpt2_model/vocab.bpe
encoder.json already exists in ../02_bonus_bytepair-encoder/gpt2_model/encoder.json


- 通过 `load_vocab_and_merges_from_openai` 方法加载文件：

In [22]:
tokenizer_gpt2 = BPETokenizerSimple()
tokenizer_gpt2.load_vocab_and_merges_from_openai(
    vocab_path=paths["encoder.json"], bpe_merges_path=paths["vocab.bpe"]
)

- 词表大小应为 `50257`，可通过以下代码确认：

In [23]:
len(tokenizer_gpt2.vocab)

50257

- 现在可通过 `BPETokenizerSimple` 对象使用 GPT-2 分词器：

In [24]:
input_text = "This is some text"
token_ids = tokenizer_gpt2.encode(input_text)
print(token_ids)

[1212, 318, 617, 2420]


In [25]:
print(tokenizer_gpt2.decode(token_ids))

This is some text


- 可用交互式 [tiktoken app](https://tiktokenizer.vercel.app/?model=gpt2) 或 [tiktoken 库](https://github.com/openai/tiktoken) 验证 token 是否正确：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/bonus/bpe-from-scratch/tiktokenizer.webp" width="600px">

```python
import tiktoken

gpt2_tokenizer = tiktoken.get_encoding("gpt2")
gpt2_tokenizer.encode("This is some text")
# 输出 [1212, 318, 617, 2420]
```


&nbsp;
# 4. 总结

- 以上就是 BPE 的要点，包含创建新分词器的训练方法，或从原始 OpenAI GPT-2 模型加载词表与 merges
- 希望本教程对教学有帮助；有问题欢迎在 [此处](https://github.com/rasbt/LLMs-from-scratch/discussions/categories/q-a) 发起讨论
- 与其他分词器实现的性能对比见 [此 notebook](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch02/02_bonus_bytepair-encoder/compare-bpe-tiktoken.ipynb)